## Installation of required library
TODO : add the version of the library for homogeneity

In [ ]:
# Run this to install libraries
!pip install --no-index nilearn
!pip install --no-index statsmodels
!pip install --no-index tqdm
!pip install --no-index seaborn
!pip install  --no-index "numpy<2"

In [ ]:
# In python by default
import sys
import glob
import os
import json
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# From other files
from utils.tools import * # To find relevant variables and wrap up
from utils.stats import * # For stat functions
from params import * # For all paths
from utils.verifications import * # For all verification steps

# Required installation
import numpy as np
import pandas as pd

### Column name

Indicate your column name and variables

In [ ]:
# Columns as indicated in your phenotype file
diagnosis_col = 'group'
subject_col = 'participant_id'
age_col = 'age'
sex_col = 'sex'
scanner_col = 'scanner'

# For dev, keep this to false
sequence_col = False
medication_col = False

# Variables in you phenotype file for case and control
case_name = 'PD-NC'
control_name = 'Control'

# and male and female
female_name = 'F'
male_name = 'M'

# Step 1: Data preparation
1. Verify the content of the phenotype file
2. Verification of all paths (altas, working directory, phenotype file, output)
3. Identify subjects who have outputs generated by HALFpipe, based on the phenotype file information
4. Reject subjects based on QC and FD (mean>0.5, max>3.0)

### Note :
Run Step 4 if you have performed QC and have a exclude.json file in your reports folder. Otherwise, keep this step as a comment.

if you have run *Step 4*, you will need to change `pheno_filtered` with `pheno_filtered_qc` in *Step 5*.

In [ ]:
# 1. Verify the phenotype file
try:
    verified_df = verify_table(pheno_p, 
                               diagnosis_col, subject_col, 
                               age_col, sex_col, scanner_col,
                               sequence_col, medication_col,
                               case_name, control_name,
                               female_name, male_name)
    print("\nPhenotype file verification successful!")
except ValueError as e:
    print(f"\nPhenotype file verification failed: {str(e)}")
    
# 2. Verify atlas location and format: only accept tsv file
try:
    verify_excepted_files()
except (FileNotFoundError, ValueError) as e:
    print(f"\nVerification failed: {str(e)}")
    
# 3. Find number of subjects processed by HALFpipe, in case some subjects failed
try:
    pheno_filtered = find_valid_subjects(verified_df)
    print(f"\nFiltered phenotype contains {len(pheno_filtered)} subjects processed by HALFpipe")
    print(f"Please verify this information is correct")

except Exception as e:
    print(f"Error: {str(e)}")
    
# 4. Reject based on bad QC
#pheno_filtered_qc = process_ratings_and_clean_phenotype(json_exclude_qc_path, pheno_filtered)

# 5. Reject subject based on FD>0.5
pheno_filtered_qc_fd = filter_subjects_by_fd(
    pheno_filtered_qc=pheno_filtered, # change pheno_filtered with pheno_filtered_qc if you have run step 4.
    derivatives_p=derivatives_p
)

## Step 2: Perform the CWAS
1. Extract relevant variable names from spec.json (atlas name, feature names, pipelines ...) 
2. Perform the CWAS for each pipeline and save the outputs

In [ ]:
labels = pd.read_csv(atlas_file, sep='\t', header=None)
conn_mask = np.tril(np.ones((len(labels),len(labels)))).astype(bool)
roi_labels = labels[1].to_list()


# Run CWAS analysis
run_cwas_analysis(
    json_file_path=json_spec_path,
    pheno_filtered_qc_fd=pheno_filtered_qc_fd,
    derivatives_p=derivatives_p,
    connectome_p=connectome_p,
    conn_mask=conn_mask,  # Your connectome mask
    roi_labels=roi_labels,  # Your ROI labels
    out_p=out_p,
    case_name=case_name,
    control_name=control_name,
    sequence_col=sequence_col, 
    medication_col=medication_col
)